<a href="https://colab.research.google.com/github/EconShark/23_big_data_analysis_class/blob/main/Lab07_Recmommendation_System_with_PySpark/Recmommendation_System_with_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THỰC HÀNH XÂY DỰNG HỆ THỐNG GỢI Ý VỚI PYSPARK

## Mục tiêu học tập

Minh họa:
- gợi ý không cá nhân hóa;
- phân tích cặp phim thường được xem cùng nhau;
- user-user / item-item collaborative filtering;
- cosine similarity;
- matrix factorization bằng SVD để minh họa;
- kiểm tra sparsity và đánh giá dự đoán.

Nội dung:
1. Phân biệt content-based filtering và collaborative filtering.
2. Phân biệt explicit rating và implicit feedback.
3. Giải thích ý tưởng ma trận User × Item và tính sparsity.
4. Hiểu trực giác matrix factorization và latent factors.
5. Chuẩn bị dữ liệu theo định dạng `userId`, `itemId`, `rating`.
6. Xây dựng mô hình ALS cho explicit ratings.
7. Giải thích các hyperparameter `rank`, `maxIter`, `regParam`, `alpha`.
8. Đánh giá ALS bằng RMSE.
9. Tuning ALS với `ParamGridBuilder` và `CrossValidator`.
10. Sinh gợi ý Top-N cho user và item.
11. Hiểu cold-start problem.
12. Làm quen với implicit ALS.
13. So sánh ALS với các kỹ thuật đơn giản như popularity, co-occurrence và cosine similarity.

<a id="setup"></a>
## Chuẩn bị môi trường


In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "pyspark": "pyspark",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib"
}

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        print(f"Đang cài {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Môi trường đã sẵn sàng.")

Môi trường đã sẵn sàng.


In [ ]:
from pathlib import Path

candidates = [
    Path.cwd() / "data",
    Path("/content/data"),
    Path.cwd().parent / "data"
]

DATA_DIR = next((p for p in candidates if p.exists()), None)

print("DATA_DIR =", DATA_DIR)

if DATA_DIR:
    for p in DATA_DIR.iterdir():
        print(" -", p.name)

DATA_DIR = /content/data
 - user_ratings.csv


## Khởi tạo SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("RecommendationEnginesWithPySpark")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark version:", spark.version)
print("Master       :", sc.master)

Spark version: 4.0.4
Master       : local[*]


<a id="overview"></a>
## 1. Tổng quan Recommendation Engine

Recommendation Engine học từ dữ liệu về:

- user;
- item;
- rating;
- lịch sử tương tác;
- hành vi xem/mua/nghe.

Ví dụ thực tế:

- phim;
- nhạc;
- sản phẩm thương mại điện tử;
- bài báo;
- khóa học;
- cổ phiếu/sản phẩm đầu tư.

Một hệ thống gợi ý cơ bản có thể được mô tả:

```text
User behavior / Ratings
          ↓
   User-Item Matrix
          ↓
 Recommendation Model
          ↓
     Predicted Score
          ↓
   Top-N Recommendations
```

<a id="types"></a>
## 2. Content-Based vs Collaborative Filtering

### Content-Based Filtering

Dựa trên **đặc trưng của item**.

Ví dụ phim:

- Genre;
- Animation;
- Language;
- Decade Produced;
- Actors.

Nếu user thích một phim, ta tìm phim khác có đặc trưng tương tự.

### Collaborative Filtering

Dựa trên **mẫu sở thích/hành vi tương tự giữa các user hoặc item**.

Ý tưởng:

> “Những user có lịch sử tương tự thường sẽ thích những item tương tự.”

ALS trong Spark là một phương pháp collaborative filtering dựa trên matrix factorization.

### So sánh nhanh

| Nội dung | Content-Based | Collaborative Filtering |
|---|---|---|
| Dữ liệu chính | Feature của item | User-item interactions |
| Cần metadata item | Có | Không bắt buộc |
| Khám phá sở thích cộng đồng | Hạn chế | Tốt |
| Cold-start user | Khó | Khó |
| Cold-start item | Có thể dùng metadata | Khó |

<a id="ratings"></a>
## 3. Explicit vs Implicit Ratings

### Explicit Ratings

User trực tiếp đưa ra đánh giá:

- 1-5 sao;
- like/dislike;
- score.

Trong dataset kèm theo, `rating` là explicit rating.

### Implicit Ratings

Không có rating trực tiếp. Ta suy ra sở thích từ hành vi:

- số lượt xem;
- số lần nghe;
- click;
- thời gian xem;
- add-to-cart;
- mua hàng.

Implicit data thường thể hiện **mức độ tin cậy** hơn là “mức độ thích” theo nghĩa rating.

<a id="matrix"></a>
## 4. Ma trận User-Item và Matrix Multiplication

Recommendation system thường được hình dung dưới dạng ma trận:

- hàng: user;
- cột: item;
- ô: rating;
- ô trống: chưa biết.

Ví dụ:

In [ ]:
import numpy as np
import pandas as pd

rating_matrix_demo = pd.DataFrame(
    [
        [5, 4, np.nan, 1],
        [4, np.nan, 5, 1],
        [1, 2, 5, np.nan],
        [np.nan, 5, 4, 4]
    ],
    index=["User1", "User2", "User3", "User4"],
    columns=["Movie1", "Movie2", "Movie3", "Movie4"]
)

rating_matrix_demo

,Movie1,Movie2,Movie3,Movie4
User1,5.0,4.0,NaN,1.0
User2,4.0,NaN,5.0,1.0
User3,1.0,2.0,5.0,NaN
User4,NaN,5.0,4.0,4.0


### Ô trống là phần cần dự đoán

Không nên đơn giản coi `NaN = 0`, vì:

```text
NaN = chưa biết / chưa rating
0   = một giá trị rating thực nếu thang đo có 0
```

### Ôn tập Matrix Multiplication

In [ ]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

B = np.array([
    [9, 8, 7],
    [6, 5, 4],
    [3, 2, 1]
])

C = A @ B

print(C)

[[ 30  24  18]
 [ 84  69  54]
 [138 114  90]]


Điều kiện nhân ma trận:

```text
A: n × m
B: m × f
→ A × B: n × f
```

Số cột của ma trận thứ nhất phải bằng số hàng của ma trận thứ hai.

<a id="factorization"></a>
## 5. Matrix Factorization và Latent Factors

Matrix factorization xấp xỉ ma trận rating:

```text
R ≈ U × V
```

Trong đó:

- `R`: ma trận user-item;
- `U`: latent factors của user;
- `V`: latent factors của item.

### Latent factor là gì?

Latent factor là đặc trưng ẩn được mô hình tự học.

Ví dụ với phim, latent factor có thể phản ánh:

- Action ↔ Drama;
- mainstream ↔ niche;
- classic ↔ modern;

nhưng ALS **không cần chúng ta đặt tên cho các factor này**.

### Rank

Nếu:

```text
U: số_user × k
V: k × số_item
```

thì `k` chính là **rank**, tức số latent factors.

Rank càng lớn:

- mô hình có khả năng biểu diễn phức tạp hơn;
- tốn tài nguyên hơn;
- có thể overfit nếu quá lớn.

### Trực giác ALS

ALS = Alternating Least Squares.

Quy trình lặp:

1. giữ user factors cố định → tối ưu item factors;
2. giữ item factors cố định → tối ưu user factors;
3. lặp lại;
4. loss/RMSE giảm dần.

Đây là ý nghĩa của từ **Alternating**.

<a id="prepare"></a>
## 6. Chuẩn bị dữ liệu cho Spark ALS

Spark ALS cần định dạng **long / row-based**:

```text
userId | movieId | rating
```

không phải dạng wide:

```text
userId | Movie A | Movie B | Movie C | ...
```

Ngoài ra `userCol` và `itemCol` phải là kiểu số nguyên.

### Đọc dataset kèm theo

In [ ]:
ratings = spark.read.csv(
    str(DATA_DIR / "user_ratings.csv"),
    header=True,
    inferSchema=True
)

ratings.show(10, truncate=False)
ratings.printSchema()

+------+-------+------+----------+----------------+-------------------------------------------+
|userId|movieId|rating|timestamp |title           |genres                                     |
+------+-------+------+----------+----------------+-------------------------------------------+
|1     |1      |4.0   |964982703 |Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|5     |1      |4.0   |847434962 |Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|7     |1      |4.5   |1106635946|Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|15    |1      |2.5   |1510577970|Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|17    |1      |4.5   |1305696483|Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|18    |1      |3.5   |1455209816|Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|19    |1      |4.0   |965705637 |Toy Story (1995)|Adventure|Animation|Children|Comedy|Fantasy|
|21    |1      |3.5   |1407618878|Toy St

Dataset kèm theo đã ở đúng dạng long và có:

- `userId`;
- `movieId`;
- `rating`;
- `timestamp`;
- `title`;
- `genres`.

Do đó không cần chuyển wide → long trước khi dùng ALS.

### Chọn đúng 3 cột cho ALS

In [ ]:
als_data = (
    ratings
    .select(
        F.col("userId").cast("int").alias("userId"),
        F.col("movieId").cast("int").alias("movieId"),
        F.col("rating").cast("float").alias("rating")
    )
    .dropna()
)

als_data.show(10)
als_data.printSchema()

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|      1|   4.0|
|     5|      1|   4.0|
|     7|      1|   4.5|
|    15|      1|   2.5|
|    17|      1|   4.5|
|    18|      1|   3.5|
|    19|      1|   4.0|
|    21|      1|   3.5|
|    27|      1|   3.0|
|    31|      1|   5.0|
+------+-------+------+
only showing top 10 rows
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)



### Ví dụ chuyển dữ liệu wide → long

Phần slide trình bày nhu cầu chuyển conventional DataFrame sang row-based format.

Trong Spark hiện đại có thể dùng `stack()`.

In [ ]:
wide_demo = spark.createDataFrame([
    ("U1", 5, 4, None),
    ("U2", 4, None, 2)
], ["userId", "MovieA", "MovieB", "MovieC"])

wide_demo.show()

+------+------+------+------+
|userId|MovieA|MovieB|MovieC|
+------+------+------+------+
|    U1|     5|     4|  NULL|
|    U2|     4|  NULL|     2|
+------+------+------+------+



In [ ]:
long_demo = wide_demo.selectExpr(
    "userId",
    "stack(3, 'MovieA', MovieA, 'MovieB', MovieB, 'MovieC', MovieC) as (title, rating)"
).filter("rating is not null")

long_demo.show()

+------+------+------+
|userId| title|rating|
+------+------+------+
|    U1|MovieA|     5|
|    U1|MovieB|     4|
|    U2|MovieA|     4|
|    U2|MovieC|     2|
+------+------+------+



<a id="eda"></a>
## 7. Khảo sát dữ liệu Movie Ratings

In [ ]:
print("Số ratings :", ratings.count())
print("Số users   :", ratings.select("userId").distinct().count())
print("Số movies  :", ratings.select("movieId").distinct().count())

Số ratings : 100836
Số users   : 610
Số movies  : 9724


### Phân bố rating

In [ ]:
ratings.groupBy("rating").count().orderBy("rating").show()

+------+-----+
|rating|count|
+------+-----+
|   0.5| 1370|
|   1.0| 2811|
|   1.5| 1791|
|   2.0| 7551|
|   2.5| 5550|
|   3.0|20047|
|   3.5|13136|
|   4.0|26818|
|   4.5| 8551|
|   5.0|13211|
+------+-----+



### Số rating theo user

In [ ]:
user_counts = (
    ratings
    .groupBy("userId")
    .count()
)

user_counts.orderBy(F.desc("count")).show(10)

+------+-----+
|userId|count|
+------+-----+
|   414| 2698|
|   599| 2478|
|   474| 2108|
|   448| 1864|
|   274| 1346|
|   610| 1302|
|    68| 1260|
|   380| 1218|
|   606| 1115|
|   288| 1055|
+------+-----+
only showing top 10 rows


### Min / Max / Avg số rating trên mỗi user

In [ ]:
user_counts.select(
    F.min("count").alias("min_ratings"),
    F.max("count").alias("max_ratings"),
    F.avg("count").alias("avg_ratings")
).show()

+-----------+-----------+------------------+
|min_ratings|max_ratings|       avg_ratings|
+-----------+-----------+------------------+
|         20|       2698|165.30491803278687|
+-----------+-----------+------------------+



### Lọc user có đủ lịch sử

In [ ]:
active_users = (
    user_counts
    .filter(F.col("count") >= 20)
)

print("Users có >= 20 ratings:", active_users.count())

Users có >= 20 ratings: 610


<a id="sparsity"></a>
## 8. Sparsity của ma trận rating

Công thức:

```text
density  = số rating quan sát / (số user × số item)
sparsity = 1 - density
```

Recommendation datasets thường rất sparse.

In [ ]:
numerator = ratings.count()
num_users = ratings.select("userId").distinct().count()
num_movies = ratings.select("movieId").distinct().count()

denominator = num_users * num_movies
density = numerator / denominator
sparsity = 1 - density

print("Ratings :", numerator)
print("Users   :", num_users)
print("Movies  :", num_movies)
print("Density :", round(density, 6))
print("Sparsity:", round(sparsity, 6))

Ratings : 100836
Users   : 610
Movies  : 9724
Density : 0.017
Sparsity: 0.983


### Ý nghĩa

Nếu sparsity gần 1:

- phần lớn cặp user-item chưa có rating;
- đây chính là lý do cần recommendation model để dự đoán các ô trống.

<a id="alsparams"></a>
## 9. ALS và các tham số

Ví dụ trong slide:

```python
ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=25,
    maxIter=100,
    regParam=0.05,
    alpha=40,
    nonnegative=True,
    coldStartStrategy="drop",
    implicitPrefs=False
)
```

### Các cột

- `userCol`: cột user ID.
- `itemCol`: cột item ID.
- `ratingCol`: cột rating/feedback.

### Hyperparameters

- `rank`: số latent factors.
- `maxIter`: số lần lặp ALS.
- `regParam`: regularization λ.
- `alpha`: mức confidence cho implicit feedback; chủ yếu có ý nghĩa khi `implicitPrefs=True`.

### Tham số bổ sung

- `nonnegative=True`: latent factors không âm.
- `coldStartStrategy="drop"`: bỏ prediction NaN cho user/item không có trong train.
- `implicitPrefs=False`: explicit rating.

<a id="basic_als"></a>
## 10. Xây dựng mô hình ALS cơ bản

Quy trình:

```text
ratings
   ↓
train/test split
   ↓
ALS()
   ↓ fit(train)
ALSModel
   ↓ transform(test)
predictions
```

In [ ]:
from pyspark.ml.recommendation import ALS

training_data, test_data = als_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Train:", training_data.count())
print("Test :", test_data.count())

Train: 80596
Test : 20240


In [ ]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=20,
    maxIter=15,
    regParam=0.1,
    nonnegative=True,
    coldStartStrategy="drop",
    implicitPrefs=False,
    seed=42
)

als_model = als.fit(training_data)
predictions = als_model.transform(test_data)

predictions.show(10)

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|    356|   4.0| 3.2095122|
|   148|  40629|   5.0| 2.5685394|
|   148|  40815|   4.0|  3.744859|
|   148|  81847|   4.5|  2.962639|
|   148| 116797|   4.5| 3.2889826|
|   148| 122886|   3.5|  3.321394|
|   148| 122920|   3.5|  2.767208|
|   463|    110|   4.5|   4.30354|
|   463|    356|   4.0| 3.9739945|
|   463|   2167|   3.0| 3.8102133|
+------+-------+------+----------+
only showing top 10 rows


<a id="rmse"></a>
## 11. Đánh giá bằng RMSE

Với explicit ratings, slide sử dụng **Root Mean Squared Error**:

```text
RMSE = sqrt(mean((prediction - actual)^2))
```

RMSE càng nhỏ càng tốt.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("RMSE =", rmse)

RMSE = 0.8701715100853348


### Xem sai số từng dự đoán

In [ ]:
(
    predictions
    .withColumn("error", F.col("prediction") - F.col("rating"))
    .withColumn("abs_error", F.abs("error"))
    .select("userId", "movieId", "rating", "prediction", "error", "abs_error")
    .orderBy(F.desc("abs_error"))
    .show(20)
)

+------+-------+------+----------+----------+---------+
|userId|movieId|rating|prediction|     error|abs_error|
+------+-------+------+----------+----------+---------+
|   105|  32892|   5.0| 0.3421859| -4.657814| 4.657814|
|   594|   2807|   5.0|  0.395816| -4.604184| 4.604184|
|   509|  84847|   4.5|0.16692111| -4.333079| 4.333079|
|   413|   4816|   0.5| 4.6102815| 4.1102815|4.1102815|
|   413|   1198|   1.0| 5.0930495| 4.0930495|4.0930495|
|   382|  70946|   0.5| 4.5690036| 4.0690036|4.0690036|
|   495|   4993|   0.5| 4.3750367| 3.8750367|3.8750367|
|    51|   1999|   5.0| 1.1464986|-3.8535013|3.8535013|
|   594|   1407|   0.5| 4.2904954| 3.7904954|3.7904954|
|   191|    673|   5.0| 1.2100339| -3.789966| 3.789966|
|     3|   7991|   5.0| 1.2111394|-3.7888606|3.7888606|
|   259|   4993|   0.5|  4.253979| 3.7539792|3.7539792|
|   536|    357|   1.0|  4.724212| 3.7242122|3.7242122|
|   564| 151315|   4.0| 0.2941595|-3.7058406|3.7058406|
|    59|   1696|   5.0| 1.3091214|-3.6908786|3.6

### Lưu ý về RMSE

RMSE đo chất lượng dự đoán rating, nhưng **không trực tiếp đo chất lượng ranking Top-N**.

Một hệ thống recommendation thực tế còn quan tâm:

- Precision@K;
- Recall@K;
- MAP/NDCG;
- Coverage;
- Diversity;
- CTR / conversion;
- A/B testing.

<a id="tuning"></a>
## 12. Hyperparameter Tuning với Cross Validation

- `ParamGridBuilder`;
- `CrossValidator`;
- `RegressionEvaluator`.

Ý tưởng:

```text
rank × maxIter × regParam
          ↓
    Cross Validation
          ↓
      Best Model
```

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

als_cv = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    implicitPrefs=False,
    seed=42
)

param_grid = (
    ParamGridBuilder()
    .addGrid(als_cv.rank, [10, 20, 40])
    .addGrid(als_cv.maxIter, [10, 20])
    .addGrid(als_cv.regParam, [0.05, 0.1, 0.2])
    .build()
)

print("Số cấu hình:", len(param_grid))

Số cấu hình: 18


In [ ]:
cv = CrossValidator(
    estimator=als_cv,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
    parallelism=2
)

# Có thể mất vài phút tùy máy.
cv_model = cv.fit(training_data)

best_model = cv_model.bestModel
best_predictions = best_model.transform(test_data)
best_rmse = evaluator.evaluate(best_predictions)

print("Best RMSE:", best_rmse)
print("Best rank:", best_model.rank)
print("Best maxIter:", best_model._java_obj.parent().getMaxIter())
print("Best regParam:", best_model._java_obj.parent().getRegParam())

### So sánh các cấu hình

In [ ]:
cv_results = []

for param_map, metric in zip(param_grid, cv_model.avgMetrics):
    params = {p.name: v for p, v in param_map.items()}
    params["RMSE_CV"] = metric
    cv_results.append(params)

cv_results_df = pd.DataFrame(cv_results).sort_values("RMSE_CV")
cv_results_df.head(10)

<a id="recommend"></a>
## 13. Sinh Recommendation cho User và Item

Spark ALS cung cấp trực tiếp:

- `recommendForAllUsers(N)`;
- `recommendForAllItems(N)`;
- `recommendForUserSubset()`;
- `recommendForItemSubset()`.

### Top-5 phim cho mỗi user

In [ ]:
user_recs = best_model.recommendForAllUsers(5)
user_recs.show(5, truncate=False)

### Nối movie title vào recommendation

In [ ]:
movie_lookup = ratings.select("movieId", "title").dropDuplicates(["movieId"])

exploded_recs = (
    user_recs
    .select("userId", F.explode("recommendations").alias("rec"))
    .select(
        "userId",
        F.col("rec.movieId").alias("movieId"),
        F.col("rec.rating").alias("predicted_rating")
    )
    .join(movie_lookup, on="movieId", how="left")
)

exploded_recs.orderBy("userId", F.desc("predicted_rating")).show(
    30,
    truncate=False
)

### Gợi ý riêng cho một user

In [ ]:
sample_user = ratings.select("userId").first()["userId"]

user_subset = spark.createDataFrame(
    [(int(sample_user),)],
    ["userId"]
)

specific_recs = best_model.recommendForUserSubset(
    user_subset,
    10
)

specific_recs.show(truncate=False)

### Top user được đề xuất cho từng item

In [ ]:
item_recs = best_model.recommendForAllItems(5)
item_recs.show(5, truncate=False)

<a id="coldstart"></a>
## 14. Cold Start và Prediction mới

Cold start xảy ra khi:

- user mới chưa có dữ liệu;
- item mới chưa có rating.

ALS không thể học latent factors cho entity chưa xuất hiện trong training set.

Vì vậy dùng:

```python
coldStartStrategy="drop"
```

để bỏ các prediction không xác định trong quá trình đánh giá.

### Minh họa

In [ ]:
known_user = training_data.select("userId").first()["userId"]

cold_start_demo = spark.createDataFrame([
    (int(known_user), 99999999)
], ["userId", "movieId"])

best_model.transform(cold_start_demo).show()

Trong production, cold-start thường được xử lý bằng:

- popularity baseline;
- content-based model;
- onboarding questions;
- hybrid recommender.

<a id="implicit"></a>
## 15. Implicit Feedback với ALS

Với implicit feedback:

```python
implicitPrefs=True
```

và `alpha` điều khiển confidence.

Dataset ZIP là explicit rating, vì vậy phần này tạo **ví dụ mở rộng** từ chính dữ liệu hiện có:

```text
interaction = 1 nếu rating >= 4
interaction = 0/không quan sát nếu rating thấp hơn
```

Đây chỉ là ví dụ giảng dạy, không phải cách duy nhất để chuyển explicit rating thành implicit feedback.

In [ ]:
implicit_data = (
    ratings
    .select("userId", "movieId", "rating")
    .withColumn(
        "preference",
        F.when(F.col("rating") >= 4.0, 1.0).otherwise(0.0)
    )
    .select(
        F.col("userId").cast("int"),
        F.col("movieId").cast("int"),
        F.col("preference").cast("float")
    )
)

implicit_data.groupBy("preference").count().show()

In [ ]:
implicit_train, implicit_test = implicit_data.randomSplit(
    [0.8, 0.2],
    seed=17
)

implicit_als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="preference",
    rank=20,
    maxIter=10,
    regParam=0.1,
    alpha=20,
    implicitPrefs=True,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=17
)

implicit_model = implicit_als.fit(implicit_train)

implicit_recs = implicit_model.recommendForAllUsers(5)
implicit_recs.show(5, truncate=False)

### Vì sao RMSE không lý tưởng cho implicit recommendation?

Trong implicit feedback:

- không tương tác không đồng nghĩa với “không thích”;
- mục tiêu thường là **ranking item phù hợp lên đầu danh sách**.

Vì vậy mở rộng sang đánh giá ranking như ROEM cho implicit feedback.  

Trong thực tế có thể dùng các metric Top-K như Precision@K, Recall@K, MAP/NDCG.

<a id="nonpersonalized"></a>
## 16. [MỞ RỘNG] Gợi ý không cá nhân hóa

Bước đơn giản nhất trước khi làm personalized recommender:

> Gợi ý các phim phổ biến và được đánh giá cao.

Một sai lầm thường gặp là chỉ xếp theo rating trung bình.  
Một phim chỉ có 1 rating = 5.0 chưa chắc đáng tin hơn phim có hàng nghìn rating = 4.5.

In [ ]:
movie_stats = (
    ratings
    .groupBy("movieId", "title")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating")
    )
)

movie_stats.orderBy(
    F.desc("avg_rating"),
    F.desc("num_ratings")
).show(20, truncate=False)

### Chỉ xét phim có đủ số lượt đánh giá

In [ ]:
popular_movies = (
    movie_stats
    .filter(F.col("num_ratings") >= 100)
    .orderBy(F.desc("avg_rating"))
)

popular_movies.show(20, truncate=False)

<a id="pairs"></a>
## 17. [MỞ RỘNG] Co-occurrence / Movie Pairs

Sử dụng `permutations()` để đếm các cặp phim được cùng user xem/rating.

Ý tưởng:

```text
User A: Movie 1, Movie 2, Movie 3
→ (1,2), (1,3), (2,1), (2,3), ...
```

Cặp xuất hiện nhiều lần có thể dùng làm gợi ý đơn giản.

In [ ]:
# Phần này dùng Pandas vì mục tiêu là minh họa đúng logic từ code ZIP.
ratings_pd = ratings.select(
    "userId", "movieId", "title", "rating"
).toPandas()

ratings_pd.head()

In [ ]:
from itertools import permutations

def create_pairs(movie_series):
    values = movie_series.drop_duplicates().values
    return list(permutations(values, 2))

pair_rows = []

# Giới hạn một phần dataset để demo nhanh nếu cần
for user_id, group in ratings_pd.groupby("userId"):
    movies = group["title"]
    for movie_a, movie_b in create_pairs(movies):
        pair_rows.append((movie_a, movie_b))

movie_pairs = pd.DataFrame(
    pair_rows,
    columns=["movie_a", "movie_b"]
)

pair_counts = (
    movie_pairs
    .value_counts()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

pair_counts.head(20)

### Hàm gợi ý theo phim

In [ ]:
def recommend_by_pair(movie_title, top_n=10):
    return (
        pair_counts[pair_counts["movie_a"] == movie_title]
        .sort_values("count", ascending=False)
        .head(top_n)
    )

recommend_by_pair("Toy Story (1995)", 10)

<a id="cosine"></a>
## 18. [MỞ RỘNG] Cosine Similarity và User/Item Collaborative Filtering

Minh họa:

- pivot user × movie;
- center rating theo user;
- fill missing bằng 0 sau khi center;
- cosine similarity;
- tìm nearest users/items.

Đây là một hướng collaborative filtering khác ALS.

In [ ]:
pivot = ratings_pd.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

pivot.iloc[:5, :8]

### Center rating theo user

In [ ]:
user_mean = pivot.mean(axis=1)

centered = pivot.sub(
    user_mean,
    axis=0
)

centered_filled = centered.fillna(0)

centered_filled.iloc[:5, :8]

### Item-item cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_matrix = centered_filled.T

# Giới hạn minh họa trên các phim có đủ ratings để giảm ma trận
movie_freq = ratings_pd["title"].value_counts()
selected_titles = movie_freq[movie_freq >= 50].index

item_matrix_small = item_matrix.loc[
    item_matrix.index.intersection(selected_titles)
]

similarities = cosine_similarity(item_matrix_small)

item_similarity_df = pd.DataFrame(
    similarities,
    index=item_matrix_small.index,
    columns=item_matrix_small.index
)

item_similarity_df.iloc[:5, :5]

### Tìm phim tương tự

In [ ]:
def similar_movies(title, top_n=10):
    if title not in item_similarity_df.index:
        return pd.Series(dtype=float)

    scores = (
        item_similarity_df.loc[title]
        .drop(title)
        .sort_values(ascending=False)
    )

    return scores.head(top_n)

similar_movies("Toy Story (1995)", 10)

### User-user similarity

In [ ]:
user_sim = cosine_similarity(centered_filled)

user_similarity_df = pd.DataFrame(
    user_sim,
    index=centered_filled.index,
    columns=centered_filled.index
)

user_similarity_df.iloc[:5, :5]

<a id="svd"></a>
## 19. [MỞ RỘNG] Matrix Factorization bằng SVD

Dùng SciPy SVD để minh họa trực tiếp matrix factorization.

Quy trình:

1. pivot user × item;
2. tính mean rating của user;
3. center rating;
4. fill missing bằng 0;
5. SVD;
6. tái tạo prediction matrix;
7. cộng lại user mean.

Đây là minh họa toán học hữu ích trước khi dùng ALS ở quy mô lớn.

### Chọn tập phim phổ biến để SVD chạy nhanh

In [ ]:
top_titles = (
    ratings_pd["title"]
    .value_counts()
    .head(300)
    .index
)

svd_df = (
    ratings_pd[ratings_pd["title"].isin(top_titles)]
    .pivot_table(
        index="userId",
        columns="title",
        values="rating"
    )
)

print("Matrix shape:", svd_df.shape)
print("Sparsity:", svd_df.isna().sum().sum() / svd_df.size)

### Center và SVD

In [ ]:
from scipy.sparse.linalg import svds

avg_ratings = svd_df.mean(axis=1)

centered_svd = (
    svd_df
    .sub(avg_ratings, axis=0)
    .fillna(0)
)

k = min(20, min(centered_svd.shape) - 1)

U, sigma, Vt = svds(
    centered_svd.values,
    k=k
)

sigma = np.diag(sigma)

pred_centered = U @ sigma @ Vt

predicted_ratings = (
    pred_centered
    + avg_ratings.values.reshape(-1, 1)
)

svd_predictions = pd.DataFrame(
    predicted_ratings,
    index=svd_df.index,
    columns=svd_df.columns
)

svd_predictions.iloc[:5, :5]

### Đánh giá trên các ô đã có rating

In [ ]:
from sklearn.metrics import mean_squared_error

actual = svd_df.values
predicted = svd_predictions.values

mask = ~np.isnan(actual)

rmse_svd = np.sqrt(
    mean_squared_error(
        actual[mask],
        predicted[mask]
    )
)

print("SVD reconstruction RMSE:", rmse_svd)

> Đây là reconstruction error trên các giá trị đã quan sát, không thay thế cho train/test evaluation thực sự.  
> ALS ở phần cốt lõi là phương pháp phù hợp hơn với workflow Spark recommendation.

<a id="project"></a>
## BÀI TẬP THÊM

### Bài toán

Xây dựng hệ thống gợi ý phim từ `user_ratings.csv`.

#### Yêu cầu tối thiểu

1. Khảo sát:
   - số users;
   - số movies;
   - số ratings;
   - sparsity.
2. Chia train/test.
3. Xây dựng ALS baseline.
4. Đánh giá RMSE.
5. Tuning:
   - `rank`;
   - `maxIter`;
   - `regParam`.
6. Chọn best model.
7. Sinh Top-10 recommendations cho ít nhất 5 users.
8. Nối `movieId` với `title`.
9. So sánh với một baseline:
   - popularity;
   - item-item cosine similarity.
10. Phân tích cold-start problem.

#### Sản phẩm nên trình bày

- Pipeline/flowchart;
- bảng thống kê dữ liệu;
- bảng hyperparameter;
- RMSE;
- Top-N recommendations;
- nhận xét ưu/nhược điểm của mô hình.

<a id="exercise"></a>
## Bài tập tự luyện

### Bài 1 - Recommendation Fundamentals

Giải thích:

1. Content-based filtering là gì?
2. Collaborative filtering là gì?
3. Explicit và implicit rating khác nhau thế nào?

---

### Bài 2 - Sparsity

Tính:

```text
density
sparsity
```

cho `user_ratings.csv`.

Giải thích ý nghĩa.

---

### Bài 3 - Data Preparation

1. Kiểm tra schema.
2. Cast `userId`, `movieId` sang integer.
3. Cast `rating` sang float.
4. Kiểm tra null.

---

### Bài 4 - ALS Baseline

Dùng:

```text
rank=10
maxIter=10
regParam=0.1
```

Tính RMSE.

---

### Bài 5 - Rank

Thử:

```text
rank = 5, 10, 20, 40, 80
```

Giữ các tham số khác cố định.

Vẽ bảng Rank vs RMSE.

---

### Bài 6 - Regularization

Thử:

```text
regParam = 0.01, 0.05, 0.1, 0.5, 1.0
```

Nhận xét underfit/overfit.

---

### Bài 7 - Cross Validation

Dùng:

```text
rank = [10, 20, 40]
regParam = [0.05, 0.1, 0.2]
```

3-fold CV.

Tìm best model.

---

### Bài 8 - Top-N Recommendation

Chọn 10 users bất kỳ.

Sinh Top-5 recommendation cho từng user và hiển thị title.

---

### Bài 9 - Popularity Baseline

Chỉ xét phim có ít nhất 100 ratings.

Xếp theo rating trung bình và so sánh với ALS.

---

### Bài 10 - Similar Movies

Dùng item-item cosine similarity.

Tìm 10 phim tương tự:

```text
Toy Story (1995)
```

So sánh với các recommendation từ ALS.

---

### Bài 11 - Implicit ALS

Chuyển rating ≥ 4 thành positive interaction.

Thử:

```text
alpha = 10, 20, 40
```

Nhận xét kết quả.

---

### Bài 12 - Cold Start

Tạo:

- user mới;
- movie mới.

Kiểm tra prediction ALS và đề xuất phương án xử lý.

<a id="summary"></a>
## Tổng kết

Chuỗi kiến thức của notebook:

```text
Recommendation Engine
       ↓
Content-Based vs Collaborative
       ↓
Explicit vs Implicit
       ↓
User-Item Matrix
       ↓
Matrix Factorization
       ↓
Latent Factors
       ↓
Spark ALS
       ↓
RMSE
       ↓
Cross Validation / Grid Search
       ↓
Top-N Recommendation
       ↓
Cold Start
       ↓
Implicit ALS
       ↓
Baselines / Cosine / SVD
```

## Những điểm cần nhớ

1. Collaborative filtering học từ **interaction**, không nhất thiết cần metadata của item.
2. ALS dùng **matrix factorization**.
3. `rank` là số latent factors.
4. `regParam` giúp kiểm soát overfitting.
5. `coldStartStrategy="drop"` đặc biệt quan trọng khi đánh giá.
6. RMSE phù hợp cho explicit rating prediction nhưng chưa đủ để đánh giá ranking.
7. Implicit feedback cần cách diễn giải và đánh giá khác explicit ratings.
8. Recommendation production nên luôn có baseline.
9. Popularity baseline đơn giản nhưng rất hữu ích.
10. Đánh giá tốt nhất trong hệ thống thật cuối cùng vẫn liên quan đến hành vi người dùng thực.

In [ ]:
# Khi hoàn thành toàn bộ notebook:
spark.stop()